In [1]:
import pandas as pd
import re
from collections import defaultdict, Counter
from rapidfuzz.distance import Levenshtein


In [3]:
# defining input and output files 
input="credit_txn_v5.xlsx"
output="output_file.xlsx"

In [5]:
# Helper function to normalise ledger name and Use levenhestein approach to find similarity 

def normalize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text.strip()

def similarity(a, b):
    a, b = normalize(a), normalize(b)
    max_len = max(len(a), len(b))
    if max_len == 0:
        return 100
    return (1 - Levenshtein.distance(a, b) / max_len) * 100

def blocking_key(text):
    text = normalize(text)
    return text[:4]   # blocking prefix


In [6]:
# forming clusters using Levehestein approach

def levenshtein_cluster_by_frequency_fast(ledger_names, threshold=82):
    freq = Counter(ledger_names)

    unique_ledgers = sorted(
        freq.keys(),
        key=lambda x: freq[x],
        reverse=True
    )

    blocks = defaultdict(list)
    for ledger in unique_ledgers:
        blocks[blocking_key(ledger)].append(ledger)

    visited = {}
    category_map = {}

    for block_ledgers in blocks.values():
        n = len(block_ledgers)

        for i in range(n):
            ledger_i = block_ledgers[i]

            if visited.get(ledger_i, False):
                continue

            category_name = ledger_i
            category_map[category_name] = [ledger_i]
            visited[ledger_i] = True

            for j in range(i + 1, n):
                ledger_j = block_ledgers[j]

                if visited.get(ledger_j, False):
                    continue

                if similarity(ledger_i, ledger_j) >= threshold:
                    category_map[category_name].append(ledger_j)
                    visited[ledger_j] = True

    return category_map


In [7]:
clusters = levenshtein_cluster_by_frequency_fast(
    df["Ledger Name"].tolist(),
    threshold=82
)


In [8]:
def build_ledger_category_map(category_map):
    ledger_to_category = {}
    for category, ledgers in category_map.items():
        for ledger in ledgers:
            ledger_to_category[ledger] = category
    return ledger_to_category


In [9]:
ledger_to_category = build_ledger_category_map(clusters)


In [10]:
df["Ledger Category"] = df["Ledger Name"].map(ledger_to_category)

# safety fallback (should be rare)
df["Ledger Category"] = df["Ledger Category"].fillna(df["Ledger Name"])


In [11]:
# 1️⃣ compute ledger-name frequency
ledger_freq = df['Ledger Name'].value_counts()

# 2️⃣ find ledgers with freq < 5
low_freq_ledgers = ledger_freq[ledger_freq < 5].index

# 3️⃣ update category
df.loc[
    df['Ledger Name'].isin(low_freq_ledgers),
    'Ledger Category'
] = "OTHER"


In [12]:
import re

ILLEGAL_EXCEL_CHARS = re.compile(r'[\x00-\x1F]')

def clean_excel_text(val):
    if isinstance(val, str):
        return ILLEGAL_EXCEL_CHARS.sub('', val)
    return val

df = df.applymap(clean_excel_text)

df.to_excel(
    output,
    index=False
)


C:\Users\subha\AppData\Local\Temp\ipykernel_1048\2910059861.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_excel_text)
